# Photo-Ring Effect Consistency Tests: `exorings` vs `geotrans2`

This notebook performs an exhaustive comparison of the **Photo-Ring (PR) effect** as
computed by two independent codes:

| Code | Variable | Method |
|---|---|---|
| `exorings-basic.py` | `PR`, `logPR` | Analytical approximation (Zuluaga et al.) |
| `geotrans2.py` (`RingedSystem.calculate_PR()`) | `self.PR` | Numerical (area integration + Seager 2003) |

### Test structure
1. **Setup** — shared constants and a bridge function translating `exorings` inputs → `geotrans2`
2. **Single-point sanity check** — default parameters
3. **Parameter sweeps** — one parameter at a time
4. **Edge cases** — opaque/transparent rings, edge-on, face-on, grazing, no-ring limit
5. **2-D grids** — joint parameter space
6. **Statistical summary** — residuals, outliers, overall agreement


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, warnings, traceback
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from scipy import constants as const

warnings.filterwarnings("ignore")

# ── Physical constants (same as both codes) ──────────────────────────────────
DAY    = const.day
HOUR   = const.hour
GCONST = const.G
DEG    = np.pi / 180
RAD    = 180 / np.pi

# Solar / planetary reference values (from geotrans2)
RSUN  = 696342.0e3       # m
MSUN  = 1.98855e30       # kg
RSAT  = 58232.0e3        # m
MSAT  = 5.6846e26        # kg
AU    = 149597871.0e3    # m
LSUN  = 3.846e26         # W
TSUN  = 5778.0           # K
RJUP  = 69911.0e3        # m
MJUP  = 1.898e27         # kg

print("Constants loaded.")
print(f"  RSUN = {RSUN/1e3:.0f} km,  MSUN = {MSUN:.3e} kg")
print(f"  RSAT = {RSAT/1e3:.0f} km,  MSAT = {MSAT:.3e} kg")


## 1 — Analytical `exorings` engine (self-contained)

We reproduce the exact analytical pipeline from `exorings-basic.py` as a callable
function so it works without the `exorings` package import.


In [ ]:
def exorings_PR(rhotrue, P, b, p, fi, fe, tau, theta, ir):
    """
    Compute the Photo-Ring effect (log10 PR) using the analytical approximation
    of Zuluaga et al., exactly as in exorings-basic.py.

    Parameters
    ----------
    rhotrue : float   Stellar density [g/cm^3]
    P       : float   Orbital period [days]
    b       : float   Impact parameter [R*]
    p       : float   Planetary radius [R*]
    fi      : float   Ring interior radius [Rp]
    fe      : float   Ring exterior radius [Rp]
    tau     : float   Ring normal opacity
    theta   : float   Projected tilt [degrees]  (90 = perpendicular to orbit)
    ir      : float   Projected inclination [degrees]  (90 = edge-on)

    Returns
    -------
    dict with keys: PR, logPR, delta, pobs, T14, T23, T14p, T23p, aobs, bobs, rhoobs
    """
    # ── Semimajor axis (a/R*) ────────────────────────────────────────────────
    a = (GCONST * (rhotrue * 1e3) / (3 * np.pi) * (P * DAY)**2)**(1/3)

    # ── Orbital inclination ──────────────────────────────────────────────────
    cosiorb = b / a
    siniorb = np.sqrt(max(0.0, 1 - cosiorb**2))

    # ── External ring projected axes ─────────────────────────────────────────
    A = fe * p                         # projected semi-major axis
    B_ring = A * np.cos(ir * DEG)      # projected semi-minor axis

    # ── Transit condition ────────────────────────────────────────────────────
    hp = max(p, A * np.sin(theta * DEG), B_ring * np.cos(theta * DEG))
    if b > 1.0 - hp:
        return None  # No transit

    # ── Absorption factor ────────────────────────────────────────────────────
    cosir = np.cos(ir * DEG)
    sinir = np.sin(ir * DEG)
    beta  = 1 - np.exp(-tau / cosir) if abs(cosir) > 1e-15 else 1.0

    # ── Effective ring radii ─────────────────────────────────────────────────
    def ring_eff_r2(f):
        fc = f * cosir
        if fc > 1:
            r2 = f**2 * cosir - 1
        else:
            y  = np.sqrt(f**2 - 1) / (f * sinir) if sinir > 1e-15 else 0.0
            r2 = (f**2 * cosir * 2 / np.pi * np.arcsin(np.clip(y, -1, 1))
                  - 2 / np.pi * np.arcsin(np.clip(y * f * cosir, -1, 1)))
        return beta * r2

    ri2 = ring_eff_r2(fi)
    re2 = ring_eff_r2(fe)

    # ── Transit depth & observed radius ─────────────────────────────────────
    ARp   = np.pi * p**2 + np.pi * (re2 - ri2) * p**2
    delta = ARp / np.pi
    pobs  = np.sqrt(delta)

    # ── Contact positions ────────────────────────────────────────────────────
    xp14 = np.sqrt(max(0, (1 + p)**2 - b**2))
    xp23 = np.sqrt(max(0, (1 - p)**2 - b**2))
    xp1, xp4 = -xp14, +xp14
    xp2, xp3 = -xp23, +xp23

    def safe_sqrt(v):
        return np.sqrt(max(0, v))

    xR13 = 1 - A**2 * (np.sin(theta * DEG) - b / A)**2 * (1 - B_ring**2 / A**2)
    xR24 = 1 - A**2 * (np.sin(theta * DEG) + b / A)**2 * (1 - B_ring**2 / A**2)

    xR1 = -safe_sqrt(xR13) - A * np.cos(theta * DEG)
    xR2 = -safe_sqrt(xR24) + A * np.cos(theta * DEG)
    xR3 = +safe_sqrt(xR13) - A * np.cos(theta * DEG)
    xR4 = +safe_sqrt(xR24) + A * np.cos(theta * DEG)

    x1 = min(xp1, xR1)
    x4 = max(xp4, xR4)
    x2 = max(xp2, xR2)
    x3 = min(xp3, xR3)

    # ── Transit durations ────────────────────────────────────────────────────
    def arcsin_safe(v):
        return np.arcsin(np.clip(v, -1, 1))

    T14p = (P * DAY) * arcsin_safe((xp4 - xp1) / (a * siniorb)) / (2 * np.pi) / HOUR
    T23p = (P * DAY) * arcsin_safe((xp3 - xp2) / (a * siniorb)) / (2 * np.pi) / HOUR
    T14  = (P * DAY) * arcsin_safe((x4  - x1 ) / (a * siniorb)) / (2 * np.pi) / HOUR
    T23  = (P * DAY) * arcsin_safe((x3  - x2 ) / (a * siniorb)) / (2 * np.pi) / HOUR

    # ── Observed orbital parameters ──────────────────────────────────────────
    aobs   = 2 * (P * DAY / HOUR) / np.pi * delta**0.25 / np.sqrt(max(1e-30, T14**2 - T23**2))
    bobs   = np.sqrt(max(0, (T14**2 * (1 - np.sqrt(delta)) - T23**2 * (1 + np.sqrt(delta)))
                           / max(1e-30, T14**2 - T23**2)))
    rhoobs = (3 * np.pi / GCONST) * aobs**3 / (P * DAY)**2 / 1e3

    PR    = rhoobs / rhotrue
    logPR = np.log10(PR)

    return dict(PR=PR, logPR=logPR, delta=delta, pobs=pobs,
                T14=T14, T23=T23, T14p=T14p, T23p=T23p,
                aobs=aobs, bobs=bobs, rhoobs=rhoobs, a=a,
                rhotrue=rhotrue)

# Quick self-test with default parameters from exorings-basic.py
_def = exorings_PR(1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0, 30.0, 80.0)
print("exorings self-test (default params):")
for k, v in _def.items():
    print(f"  {k:12s} = {v:.6g}")


## 2 — `geotrans2` bridge: exorings inputs → `RingedSystem`

`geotrans2` requires physical units (Mstar, Rstar, Rplanet, ap, …) whereas `exorings`
works in stellar-radius units with stellar density as input.  The bridge below converts
the `exorings` parameter set to the physical quantities needed by `RingedSystem`.

Key conversions:
- `Rstar` from `Mstar` via mass-radius relation `R* ∝ M*^0.8`  (same as geotrans2 defaults)
- `Rplanet = p * Rstar`
- `ap` from Kepler's third law using `rhotrue` and `P`
- `iorb` from impact parameter: `cos(iorb) = b / (a/R*)`
- Ring angles: `ir` → geotrans2 `ir`; `theta` → geotrans2 `phir`

> **Note on `phir` vs `theta`**: In `exorings`, `theta` is the projected ring tilt
> on the sky plane. In `geotrans2`, the effective tilt (`teff`) is a 3-D projection
> of `phir` and `ir` through the orbital inclination matrix.  For the circular orbit
> (e=0, iorb≈90°) case treated by `exorings`, `teff ≈ theta` when `phir` is chosen
> appropriately. We enforce this analytically below.


In [ ]:
import os, sys
import importlib.util

# Expand the ~ to the full user directory
GEOTRANS_DIR = os.path.expanduser("~/kepler_51/Zuluaga_PhotoRing/GeoTrans/")
GEOTRANS_PATH = os.path.join(GEOTRANS_DIR, "geotrans2.py")

spec = importlib.util.spec_from_file_location("geotrans2", GEOTRANS_PATH)
gt2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gt2)

RingedSystem = gt2.RingedSystem
print("geotrans2 loaded.")
print(f"  RingedSystem attributes sample: Mstar, Rplanet, fe, fi, tau, ir, phir")


# ── Conversion function ───────────────────────────────────────────────────────
def exorings_to_geotrans(rhotrue, P, b, p, fi, fe, tau, theta, ir,
                          Mstar_solar=1.0):
    """
    Build a geotrans2.RingedSystem from exorings-style inputs.

    Parameters
    ----------
    rhotrue      : stellar density [g/cm^3]
    P            : orbital period [days]
    b            : impact parameter [R*]
    p            : planet radius [R*]
    fi, fe       : ring inner/outer radii [Rp]
    tau          : ring normal opacity
    theta        : projected ring tilt [deg] (exorings convention)
    ir           : projected ring inclination [deg] (90 = edge-on)
    Mstar_solar  : stellar mass in solar masses (default 1.0)

    Returns
    -------
    RingedSystem instance  or  None if no transit
    """
    # Stellar properties
    Mstar = Mstar_solar * MSUN
    # Use geotrans2's own mass-radius relation (R* ∝ M*^0.8)
    Rstar = RSUN * (Mstar / MSUN)**0.8

    # Verify rhotrue consistency: rho = M / (4/3 pi R^3)
    rho_check = Mstar / (4 * np.pi / 3 * Rstar**3) / 1e3   # g/cm^3
    # If user supplies a different rhotrue, scale Mstar accordingly
    # (keep Rstar fixed, adjust Mstar so density matches)
    Mstar_adjusted = rhotrue * 1e3 * (4 * np.pi / 3 * Rstar**3)

    # Semimajor axis from rhotrue and P (Kepler + stellar density)
    a_over_Rstar = (GCONST * (rhotrue * 1e3) / (3 * np.pi) * (P * DAY)**2)**(1/3)
    ap = a_over_Rstar * Rstar       # meters

    # Planetary radius
    Rplanet = p * Rstar

    # Orbital inclination from impact parameter
    cosiorb = b / a_over_Rstar
    if abs(cosiorb) > 1.0:
        return None   # no transit possible
    iorb = np.arccos(cosiorb)        # radians  (close to pi/2 for near-transit)

    # ── Ring angles ──────────────────────────────────────────────────────────
    # exorings theta is the projected tilt of the ring on the sky.
    # geotrans2 uses ir (inclination wrt orbital plane) + phir (roll/azimuth).
    #
    # For a circular, near-edge-on orbit (iorb ≈ 90°, e=0):
    #   ieff  ≈ ir_geotrans  (ring inclination as seen by the observer)
    #   teff  ≈ theta         (projected tilt on sky)
    #
    # To reproduce the exorings theta:
    #   We set phir such that teff = theta.
    #   For iorb ≈ 90°, phir ≈ theta is a good first approximation.
    #   We use phir = theta (in radians), ir_geotrans = ir (in radians).
    ir_rad   = ir    * DEG
    phir_rad = theta * DEG   # approximate; exact for iorb=90

    try:
        S = RingedSystem(dict(
            Mstar   = Mstar_adjusted,
            Rstar   = Rstar,
            Mplanet = MSAT,              # mass doesn't affect transit geometry
            Rplanet = Rplanet,
            ap      = ap,
            ep      = 0.0,
            iorb    = iorb,
            wp      = 0.0 * DEG,
            fi      = fi,
            fe      = fe,
            tau     = tau,
            ir      = ir_rad,
            phir    = phir_rad,
            fp      = 0.0,
        ))
        return S
    except SystemExit:
        return None
    except Exception as exc:
        return None


# ── Test bridge with default params ─────────────────────────────────────────
S_def = exorings_to_geotrans(1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0, 30.0, 80.0)
if S_def is not None:
    PR_geo = S_def.calculate_PR()
    print(f"geotrans2  logPR (default) = {PR_geo:.6f}")
    print(f"exorings   logPR (default) = {_def['logPR']:.6f}")
    print(f"Difference (Δ)             = {abs(PR_geo - _def['logPR']):.6f}")
else:
    print("WARNING: bridge returned None for default parameters")


## 3 — Helper utilities

In [ ]:
def compare_PR(rhotrue, P, b, p, fi, fe, tau, theta, ir, label=""):
    """
    Run both codes and return a comparison record.
    Returns a dict with analytical result, numerical result, difference, and status.
    """
    result = dict(
        rhotrue=rhotrue, P=P, b=b, p=p,
        fi=fi, fe=fe, tau=tau, theta=theta, ir=ir,
        label=label,
        logPR_exo=np.nan, logPR_geo=np.nan, delta=np.nan,
        status="OK"
    )

    # Numerical (geotrans2)
    try:
        S = exorings_to_geotrans(rhotrue, P, b, p, fi, fe, tau, theta, ir)
        if S is None:
            result["status"] = "NO_TRANSIT_GEO"
            return result
        logPR_geo = S.calculate_PR()
        result["logPR_geo"] = logPR_geo
        # result["ieff_deg"]  = S.ieff * RAD
        # result["teff_deg"]  = S.teff * RAD
        # ── observables ───────────────────────────────────────
        result['delta_geo']  = gt2.ringedPlanetArea(S) / np.pi
        # gt_Ana    = analyticalTransitAreaSystem(S) / np.pi
        # ref_Ana   = ref['ARp'] / np.pi
        try:
            tcsp = gt2.contactTimes(S)
            tT   = (tcsp[-1]-tcsp[1])/HOUR
            tF   = (tcsp[-2]-tcsp[2])/HOUR
            result["T14_geo"] = tT
            result["T23_geo"] = tF
        except Exception:
            tT = tF = np.nan
    except Exception as e:
        result["status"] = f"GEO_ERR:{e}"

    # Analytical (exorings)
    try:
        exo = exorings_PR(rhotrue, P, b, p, fi, fe, tau, S.teff*RAD, S.ieff*RAD)
        if exo is None:
            result["status"] = "NO_TRANSIT_EXO"
            return result
        result["logPR_exo"] = exo["logPR"]
        result["delta_exo"]     = exo["delta"]
        result["T14_exo"]       = exo["T14"]
        result["T23_exo"]       = exo["T23"]
    except Exception as e:
        result["status"] = f"EXO_ERR:{e}"
        return result

    result["diff"]    = result["logPR_geo"] - result["logPR_exo"]
    result["rel_err"] = abs(result["diff"]) / (abs(result["logPR_exo"]) + 1e-10)
    return result


def fmt_row(r):
    if np.isnan(r.get("logPR_exo", np.nan)):
        return f"[{r['status']}]"
    d = r.get("diff", np.nan)
    return (f"logPR_exo={r['logPR_exo']:+.4f}  "
            f"logPR_geo={r['logPR_geo']:+.4f}  "
            f"Δ={d:+.4f}  [{r['status']}]")


print("Helper utilities defined.")


## 4 — Single-point sanity check (default parameters)

Reproduce the exact parameters from `exorings-basic.py` header.


In [ ]:
# Default parameters exactly as in exorings-basic.py
defaults = dict(rhotrue=1.40598, P=365.2446, b=0.1875,
                p=0.08, fi=1.5, fe=2.35, tau=1.0, theta=30.0, ir=80.0)

r = compare_PR(**defaults, label="DEFAULT")
print("=" * 65)
print(f"  Label      : {r['label']}")
print(f"  logPR_exo  : {r['logPR_exo']:+.6f}")
print(f"  logPR_geo  : {r['logPR_geo']:+.6f}")
print(f"  Δ          : {r['diff']:+.6f}")
print(f"  Rel. error : {r['rel_err']:.4%}")
print(f"  Status     : {r['status']}")
print(f"\n  Geotrans2\n")
print(f"  δ (ppm)    : {r['delta_geo']*1e6:.1f}")
print(f"  T14 (h)    : {r['T14_geo']:.4f}")
print(f"  T23 (h)    : {r['T23_geo']:.4f}")
print(f"\n  Exorings\n")
print(f"  δ (ppm)    : {r['delta_exo']*1e6:.1f}")
print(f"  T14 (h)    : {r['T14_exo']:.4f}")
print(f"  T23 (h)    : {r['T23_exo']:.4f}")
print("=" * 65)


## 5 — 1-D Parameter sweeps

Each panel sweeps one parameter while holding the others at their default values.
We plot:
- **logPR_exo** (analytical, blue)
- **logPR_geo** (numerical, orange)
- **Δ = logPR_geo − logPR_exo** (green, right axis)


In [ ]:
# Default parameter dict
DEF = dict(rhotrue=1.40598, P=365.2446, b=0.1875,
           p=0.08, fi=1.5, fe=2.35, tau=1.0, theta=30.0, ir=80.0)

# Parameter sweep definitions: (key, values, label, unit)
sweeps = [
    ("ir",      np.linspace(10, 89, 40),        r"Ring inclination $i_r$", "deg"),
    ("theta",   np.linspace(1,  89, 40),         r"Projected tilt $\theta$", "deg"),
    ("fe",      np.linspace(1.1, 5.0, 40),       r"Exterior ring $f_e$", r"$R_p$"),
    ("fi",      np.linspace(1.0, 2.5, 35),       r"Interior ring $f_i$", r"$R_p$"),
    ("tau",     np.linspace(0.01, 5.0, 40),      r"Opacity $\tau$", ""),
    ("b",       np.linspace(0.0, 0.6, 40),       r"Impact parameter $b$", r"$R_*$"),
    ("p",       np.linspace(0.04, 0.20, 35),     r"Planet radius $p$", r"$R_*$"),
    ("P",       np.linspace(50,  500, 40),        r"Orbital period $P$", "days"),
    ("rhotrue", np.linspace(0.3, 5.0, 40),       r"Stellar density $\rho_*$", r"g/cm³"),
]

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()
sweep_results = {}

for ax_idx, (key, vals, lbl, unit) in enumerate(sweeps):
    ax  = axes[ax_idx]
    ax2 = ax.twinx()

    exo_vals, geo_vals, diffs, xs_ok = [], [], [], []

    for v in vals:
        kw = dict(DEF)
        kw[key] = v
        # Guard: fi must be < fe
        if key == "fi" and v >= kw["fe"]:
            continue
        if key == "fe" and v <= kw["fi"]:
            continue
        r = compare_PR(**kw)
        if r["status"] == "OK":
            exo_vals.append(r["logPR_exo"])
            geo_vals.append(r["logPR_geo"])
            diffs.append(r["diff"])
            xs_ok.append(v)

    sweep_results[key] = dict(xs=xs_ok, exo=exo_vals, geo=geo_vals, diffs=diffs)

    if xs_ok:
        ax.plot(xs_ok, exo_vals, "b-o", ms=3, lw=1.5, label="exorings (analytic)")
        ax.plot(xs_ok, geo_vals, "r--s", ms=3, lw=1.5, label="geotrans2 (numeric)")
        ax2.plot(xs_ok, diffs, "g:", lw=1.2, alpha=0.8, label=r"$\Delta$")
        ax2.axhline(0, color="gray", lw=0.7, ls="--")
        ax2.set_ylabel(r"$\Delta \log PR$", color="g", fontsize=8)
        ax2.tick_params(axis="y", labelcolor="g", labelsize=7)

    xlabel = f"{lbl}" + (f" [{unit}]" if unit else "")
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(r"$\log_{10}(\rho_{\rm obs}/\rho_{\rm true})$", fontsize=9)
    ax.set_title(lbl, fontsize=10)
    ax.legend(fontsize=7, loc="best")
    ax.grid(True, alpha=0.3)

fig.suptitle("1-D Parameter Sweeps: logPR comparison (exorings vs geotrans2)",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("/tmp/sweep1d.png", dpi=120, bbox_inches="tight")
plt.show()
print("1-D sweeps done.")


## 6 — Edge cases

Critical configurations that test the boundary conditions of both models.


In [ ]:
edge_cases = [
    # label, rhotrue, P, b, p, fi, fe, tau, theta, ir
    ("No rings (tau→0)",           1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 0.001, 30.0, 80.0),
    ("Opaque rings (tau=10)",      1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 10.0,  30.0, 80.0),
    ("Edge-on rings (ir=89°)",     1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 89.0),
    ("Near face-on (ir=10°)",      1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 10.0),
    ("Central transit (b=0)",      1.40598, 365.2446, 0.0,    0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("High-b transit (b=0.5)",     1.40598, 365.2446, 0.50,   0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Small planet (p=0.04)",      1.40598, 365.2446, 0.1875, 0.04, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Large planet (p=0.15)",      1.40598, 365.2446, 0.1875, 0.15, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Wide rings (fe=5)",          1.40598, 365.2446, 0.1875, 0.08, 1.5, 5.0,  1.0,   30.0, 80.0),
    ("Narrow ring gap (fi=2.0)",   1.40598, 365.2446, 0.1875, 0.08, 2.0, 2.35, 1.0,   30.0, 80.0),
    ("Theta=0 (aligned rings)",    1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   1.0,  80.0),
    ("Theta=89 (perpend. rings)",  1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   89.0, 80.0),
    ("Short period (P=10d)",       1.40598, 10.0,     0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Long period (P=1000d)",      0.50,    1000.0,   0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Dense star (rho=5 g/cc)",    5.0,     365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Rarefied star (rho=0.3)",    0.3,     365.2446, 0.1875, 0.08, 1.5, 2.35, 1.0,   30.0, 80.0),
    ("Saturn-like system",         1.40598, 29.46*365, 0.1, 0.0838, 1.58, 2.35, 1.0, 45.0, 75.0),
    ("Partial tau (tau=0.5)",      1.40598, 365.2446, 0.1875, 0.08, 1.5, 2.35, 0.5,   30.0, 80.0),
]

print(f"{'Label':<32} {'logPR_exo':>10} {'logPR_geo':>10} {'Δ':>8} {'Status'}")
print("-" * 75)
edge_records = []
for ec in edge_cases:
    label, rhotrue, P, b, p, fi, fe, tau, theta, ir = ec
    r = compare_PR(rhotrue, P, b, p, fi, fe, tau, theta, ir, label=label)
    edge_records.append(r)
    exo_s = f"{r['logPR_exo']:+.4f}" if not np.isnan(r['logPR_exo']) else "  N/A  "
    geo_s = f"{r['logPR_geo']:+.4f}" if not np.isnan(r['logPR_geo']) else "  N/A  "
    d_s   = f"{r.get('diff', np.nan):+.4f}" if not np.isnan(r.get('diff', np.nan)) else "  N/A  "
    print(f"{label:<32} {exo_s:>10} {geo_s:>10} {d_s:>8} {r['status']}")


## 7 — 2-D parameter grids

The most physically informative plane is **(ir, theta)** — the ring geometry space.
We also explore **(fe, ir)** and **(b, ir)**.

Each panel shows `Δ = logPR_geo − logPR_exo` as a colour map (diverging around 0).


In [ ]:
def sweep_2d(key1, vals1, key2, vals2, base=None, n_skip_msg=True):
    """Run a 2-D sweep and return arrays for plotting."""
    if base is None:
        base = dict(DEF)
    N1, N2 = len(vals1), len(vals2)
    exo_grid  = np.full((N2, N1), np.nan)
    geo_grid  = np.full((N2, N1), np.nan)
    diff_grid = np.full((N2, N1), np.nan)

    n_ok = n_fail = 0
    for i, v1 in enumerate(vals1):
        for j, v2 in enumerate(vals2):
            kw = dict(base)
            kw[key1] = v1
            kw[key2] = v2
            if "fi" in kw and "fe" in kw and kw["fi"] >= kw["fe"]:
                n_fail += 1
                continue
            r = compare_PR(**kw)
            if r["status"] == "OK":
                exo_grid[j, i]  = r["logPR_exo"]
                geo_grid[j, i]  = r["logPR_geo"]
                diff_grid[j, i] = r["diff"]
                n_ok += 1
            else:
                n_fail += 1
    if n_skip_msg:
        print(f"  Grid {key1} x {key2}: {n_ok} OK, {n_fail} skipped/failed")
    return exo_grid, geo_grid, diff_grid


# ── Define 2-D grids ──────────────────────────────────────────────────────────
N = 30
grids_2d = [
    ("ir",    np.linspace(10, 89, N),  "theta", np.linspace(1, 89, N),
     r"Ring inclination $i_r$ [°]",    r"Projected tilt $\theta$ [°]"),

    ("fe",    np.linspace(1.1, 5.0, N), "ir",   np.linspace(10, 89, N),
     r"Exterior ring $f_e$ [$R_p$]",   r"Ring inclination $i_r$ [°]"),

    ("b",     np.linspace(0.0, 0.55, N), "ir",  np.linspace(10, 89, N),
     r"Impact parameter $b$ [$R_*$]",   r"Ring inclination $i_r$ [°]"),

    ("tau",   np.linspace(0.05, 5.0, N), "ir",  np.linspace(10, 89, N),
     r"Opacity $\tau$",               r"Ring inclination $i_r$ [°]"),
]

fig, axes = plt.subplots(len(grids_2d), 3, figsize=(18, 5 * len(grids_2d)))

for row, (k1, v1, k2, v2, lbl1, lbl2) in enumerate(grids_2d):
    print(f"Computing grid {row+1}/{len(grids_2d)}: {k1} x {k2} ...")
    exo_g, geo_g, diff_g = sweep_2d(k1, v1, k2, v2)

    vmax = np.nanpercentile(np.abs(diff_g), 95)
    vmax = max(vmax, 0.05)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    extent = [v1[0], v1[-1], v2[0], v2[-1]]
    aspect = "auto"

    for col, (data, title, cmap) in enumerate([
        (exo_g,  r"$\log PR$ exorings (analytical)", "viridis"),
        (geo_g,  r"$\log PR$ geotrans2 (numerical)",  "viridis"),
        (diff_g, r"$\Delta \log PR$ (geo − exo)",    "RdBu_r"),
    ]):
        ax = axes[row, col]
        norm_use = norm if col == 2 else None
        im = ax.imshow(data, origin="lower", extent=extent,
                       aspect=aspect, cmap=cmap, norm=norm_use)
        ax.set_xlabel(lbl1, fontsize=9)
        ax.set_ylabel(lbl2, fontsize=9)
        ax.set_title(title, fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.85)

fig.suptitle("2-D Grids: logPR comparison (exorings vs geotrans2)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("/tmp/grids2d.png", dpi=110, bbox_inches="tight")
plt.show()
print("2-D grids done.")


## 8 — No-ring limit: τ → 0 and fi → fe

When opacity vanishes (τ→0) or the ring gap closes (fi→fe), both codes should
converge to **logPR = 0** (transparent rings produce no photo-ring effect).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Tau → 0
taus = np.logspace(-3, 1, 50)
r_exo, r_geo, diffs = [], [], []
for t in taus:
    r = compare_PR(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                   DEF["fi"], DEF["fe"], t, DEF["theta"], DEF["ir"])
    if r["status"] == "OK":
        r_exo.append(r["logPR_exo"])
        r_geo.append(r["logPR_geo"])
        diffs.append(r["diff"])
    else:
        r_exo.append(np.nan); r_geo.append(np.nan); diffs.append(np.nan)

ax = axes[0]
ax.semilogx(taus, r_exo, "b-o", ms=4, label="exorings")
ax.semilogx(taus, r_geo, "r--s", ms=4, label="geotrans2")
ax.axhline(0, color="k", lw=0.7, ls=":")
ax.set_xlabel(r"Opacity $\tau$", fontsize=11)
ax.set_ylabel(r"$\log_{10}(\rho_{\rm obs}/\rho_{\rm true})$", fontsize=11)
ax.set_title(r"No-ring limit: $\tau \to 0$", fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# fi → fe (closing ring gap)
fi_vals = np.linspace(1.0, DEF["fe"] - 0.01, 50)
r_exo2, r_geo2 = [], []
for fi in fi_vals:
    r = compare_PR(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                   fi, DEF["fe"], DEF["tau"], DEF["theta"], DEF["ir"])
    if r["status"] == "OK":
        r_exo2.append(r["logPR_exo"])
        r_geo2.append(r["logPR_geo"])
    else:
        r_exo2.append(np.nan); r_geo2.append(np.nan)

ax2 = axes[1]
ax2.plot(fi_vals, r_exo2, "b-o", ms=4, label="exorings")
ax2.plot(fi_vals, r_geo2, "r--s", ms=4, label="geotrans2")
ax2.axvline(DEF["fe"], color="gray", ls=":", lw=0.8, label=r"$f_i = f_e$")
ax2.set_xlabel(r"Interior ring radius $f_i$ [$R_p$]", fontsize=11)
ax2.set_ylabel(r"$\log_{10}(\rho_{\rm obs}/\rho_{\rm true})$", fontsize=11)
ax2.set_title(r"No-ring limit: $f_i \to f_e$", fontsize=12)
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle("No-ring limits", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("/tmp/noring.png", dpi=120)
plt.show()


## 9 — Monte Carlo random draw (N = 500)

Randomly sample the full parameter space and compute both PR values.
This gives a statistical picture of agreement across all configurations.


In [ ]:
np.random.seed(42)
N_MC = 500

mc_params = dict(
    rhotrue = np.random.uniform(0.3,  5.0,  N_MC),
    P       = np.random.uniform(10,   500,  N_MC),
    b       = np.random.uniform(0.0,  0.6,  N_MC),
    p       = np.random.uniform(0.04, 0.20, N_MC),
    fi      = np.random.uniform(1.0,  2.0,  N_MC),
    fe      = np.random.uniform(2.0,  5.0,  N_MC),
    tau     = np.random.uniform(0.1,  5.0,  N_MC),
    theta   = np.random.uniform(1.0,  89.0, N_MC),
    ir      = np.random.uniform(10.0, 89.0, N_MC),
)

mc_records = []
for i in range(N_MC):
    kw = {k: mc_params[k][i] for k in mc_params}
    r  = compare_PR(**kw, label=f"MC_{i}")
    mc_records.append(r)

ok_recs  = [r for r in mc_records if r["status"] == "OK"]
fail_recs = [r for r in mc_records if r["status"] != "OK"]

exo_mc   = np.array([r["logPR_exo"] for r in ok_recs])
geo_mc   = np.array([r["logPR_geo"] for r in ok_recs])
diff_mc  = np.array([r["diff"]      for r in ok_recs])

print(f"Monte Carlo: {len(ok_recs)}/{N_MC} successful, {len(fail_recs)} failed")
print(f"  Failure reasons: {set(r['status'] for r in fail_recs)}")
print()
print(f"  Δ = logPR_geo − logPR_exo:")
print(f"    Mean      = {diff_mc.mean():+.5f}")
print(f"    Std       = {diff_mc.std():.5f}")
print(f"    Median    = {np.median(diff_mc):+.5f}")
print(f"    |Δ| < 0.1 : {(np.abs(diff_mc) < 0.1).mean()*100:.1f}%")
print(f"    |Δ| < 0.5 : {(np.abs(diff_mc) < 0.5).mean()*100:.1f}%")
print(f"    |Δ| ≥ 0.5 : {(np.abs(diff_mc) >= 0.5).mean()*100:.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) scatter plot
ax = axes[0]
ax.scatter(exo_mc, geo_mc, s=6, alpha=0.5, c=np.abs(diff_mc), cmap="plasma")
lo = min(exo_mc.min(), geo_mc.min()); hi = max(exo_mc.max(), geo_mc.max())
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="1:1")
ax.set_xlabel(r"$\log PR$ exorings", fontsize=11)
ax.set_ylabel(r"$\log PR$ geotrans2", fontsize=11)
ax.set_title("Scatter: exorings vs geotrans2", fontsize=11)
ax.legend(); ax.grid(True, alpha=0.3)

# 2) histogram of Δ
ax2 = axes[1]
ax2.hist(diff_mc, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
ax2.axvline(0, color="k", ls="--", lw=1.5)
ax2.axvline(diff_mc.mean(), color="r", ls=":", lw=1.5, label=f"mean={diff_mc.mean():+.4f}")
ax2.set_xlabel(r"$\Delta \log PR$ (geo − exo)", fontsize=11)
ax2.set_ylabel("Count", fontsize=11)
ax2.set_title(r"Distribution of $\Delta \log PR$", fontsize=11)
ax2.legend(); ax2.grid(True, alpha=0.3)

# 3) |Δ| vs logPR_exo
ax3 = axes[2]
ax3.scatter(exo_mc, np.abs(diff_mc), s=6, alpha=0.4, c="darkorange")
ax3.axhline(0.1, color="b", ls="--", lw=1, label="|Δ|=0.1")
ax3.axhline(0.5, color="r", ls="--", lw=1, label="|Δ|=0.5")
ax3.set_xlabel(r"$\log PR$ exorings", fontsize=11)
ax3.set_ylabel(r"|$\Delta \log PR$|", fontsize=11)
ax3.set_title("Absolute difference vs logPR", fontsize=11)
ax3.legend(); ax3.grid(True, alpha=0.3)

fig.suptitle(f"Monte Carlo comparison (N={len(ok_recs)} valid runs)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("/tmp/mc_diagnostics.png", dpi=120)
plt.show()

# Most discrepant cases
print("\nTop 10 most discrepant cases (|Δ| largest):")
sorted_ok = sorted(ok_recs, key=lambda r: abs(r["diff"]), reverse=True)
print(f"{'ir':>6} {'theta':>7} {'fe':>5} {'fi':>5} {'tau':>5} {'b':>5} "
      f"{'logPR_exo':>10} {'logPR_geo':>10} {'Δ':>8}")
for r in sorted_ok[:10]:
    print(f"{r['ir']:6.1f} {r['theta']:7.1f} {r['fe']:5.2f} {r['fi']:5.2f} "
          f"{r['tau']:5.2f} {r['b']:5.3f} "
          f"{r['logPR_exo']:10.4f} {r['logPR_geo']:10.4f} {r['diff']:+8.4f}")


## 10 — Analytical cross-check: `transitFunction` (geotrans2) vs exorings δ

`geotrans2` exposes `analyticalTransitArea()` which uses the same analytical integral
as exorings.  We verify both codes return the same transit area before the PR calculation.


In [ ]:
# analyticalTransitArea from geotrans2
analyticalTransitArea = gt2.analyticalTransitArea

def geotrans_analytic_depth(p, fi, fe, tau, ir):
    """Compute transit depth using geotrans2 analytical formula."""
    cosir = np.cos(ir * DEG)
    block = 1 - np.exp(-tau / cosir) if abs(cosir) > 1e-15 else 1.0
    A = analyticalTransitArea(p, block, fi, fe, ir * DEG)
    return A / np.pi   # delta = A / A_star (star radius = 1)

def exorings_analytic_depth(p, fi, fe, tau, ir):
    """Compute transit depth using exorings analytical formula."""
    res = exorings_PR(1.40598, 365.2446, 0.1875, p, fi, fe, tau, 30.0, ir)
    if res is None:
        return np.nan
    return res["delta"]

ir_vals = np.linspace(10, 89, 60)
dep_exo, dep_geo, dep_diff = [], [], []

for ir_v in ir_vals:
    d_exo = exorings_analytic_depth(DEF["p"], DEF["fi"], DEF["fe"], DEF["tau"], ir_v)
    d_geo = geotrans_analytic_depth(DEF["p"], DEF["fi"], DEF["fe"], DEF["tau"], ir_v)
    dep_exo.append(d_exo)
    dep_geo.append(d_geo)
    dep_diff.append(d_geo - d_exo)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(ir_vals, np.array(dep_exo)*1e6, "b-o", ms=4, label="exorings")
ax1.plot(ir_vals, np.array(dep_geo)*1e6, "r--s", ms=4, label="geotrans2 (analytic)")
ax1.set_xlabel(r"Ring inclination $i_r$ [°]", fontsize=11)
ax1.set_ylabel("Transit depth δ [ppm]", fontsize=11)
ax1.set_title("Transit depth: analytical cross-check", fontsize=12)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ir_vals, np.array(dep_diff)*1e6, "g-o", ms=4)
ax2.axhline(0, color="k", ls="--", lw=0.8)
ax2.set_xlabel(r"Ring inclination $i_r$ [°]", fontsize=11)
ax2.set_ylabel("Δδ [ppm]  (geo − exo)", fontsize=11)
ax2.set_title("Δ transit depth (analytical codes)", fontsize=12)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig("/tmp/analytic_depth.png", dpi=120)
plt.show()

print(f"Max |Δδ| = {np.nanmax(np.abs(dep_diff))*1e6:.3f} ppm")
print(f"Mean |Δδ| = {np.nanmean(np.abs(dep_diff))*1e6:.3f} ppm")


## 11 — Physical consistency diagnostics

Check that both codes satisfy expected physical inequalities:
- PR ≥ 1 always (ringed planet always looks denser)
- PR → 1 as τ → 0
- PR increases with wider/more opaque rings


In [ ]:
# Test: logPR >= 0 always (physical constraint)
exo_neg = exo_mc[exo_mc < -0.001]
geo_neg = geo_mc[geo_mc < -0.001]
print(f"Physical constraint (logPR >= 0):")
print(f"  exorings  violations: {len(exo_neg)}/{len(exo_mc)}  "
      f"(min={exo_mc.min():.4f})")
print(f"  geotrans2 violations: {len(geo_neg)}/{len(geo_mc)}  "
      f"(min={geo_mc.min():.4f})")

# Test: monotonicity in tau
print("\nMonotonicity in tau (logPR should increase with tau):")
taus = np.linspace(0.01, 8.0, 40)
lr_exo = [exorings_PR(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                       DEF["fi"], DEF["fe"], t, DEF["theta"], DEF["ir"])["logPR"]
          for t in taus]
S_list = [exorings_to_geotrans(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                                DEF["fi"], DEF["fe"], t, DEF["theta"], DEF["ir"])
          for t in taus]
lr_geo = [S.calculate_PR() if S else np.nan for S in S_list]

diffs_exo = np.diff(lr_exo)
diffs_geo = np.diff(lr_geo)
print(f"  exorings  monotone increases: {(diffs_exo > 0).sum()}/{len(diffs_exo)}")
print(f"  geotrans2 monotone increases: {(np.array(diffs_geo) > 0).sum()}/{len(diffs_geo)}")

# Test: logPR → 0 as tau → 0
r_tau0_exo = exorings_PR(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                          DEF["fi"], DEF["fe"], 1e-4, DEF["theta"], DEF["ir"])["logPR"]
S_tau0 = exorings_to_geotrans(DEF["rhotrue"], DEF["P"], DEF["b"], DEF["p"],
                               DEF["fi"], DEF["fe"], 1e-4, DEF["theta"], DEF["ir"])
r_tau0_geo = S_tau0.calculate_PR() if S_tau0 else np.nan
print(f"\nLimit tau→0 (tau=1e-4):")
print(f"  logPR_exo = {r_tau0_exo:.6f}  (expected ≈ 0)")
print(f"  logPR_geo = {r_tau0_geo:.6f}  (expected ≈ 0)")


## 12 — Summary report


In [ ]:
print("=" * 70)
print("  PHOTO-RING CONSISTENCY REPORT: exorings vs geotrans2")
print("=" * 70)

# Default single point
r0 = compare_PR(**DEF, label="Default")
print(f"\n[DEFAULT PARAMS]")
print(f"  logPR exorings  = {r0['logPR_exo']:+.6f}")
print(f"  logPR geotrans2 = {r0['logPR_geo']:+.6f}")
print(f"  |Δ|             = {abs(r0['diff']):.6f}")

# Edge cases summary
ok_edge  = [r for r in edge_records if r["status"] == "OK"]
fail_edge = [r for r in edge_records if r["status"] != "OK"]
diffs_edge = [r["diff"] for r in ok_edge]
print(f"\n[EDGE CASES]")
print(f"  Total: {len(edge_records)}  |  OK: {len(ok_edge)}  |  Failed: {len(fail_edge)}")
if diffs_edge:
    print(f"  |Δ| mean  = {np.mean(np.abs(diffs_edge)):.5f}")
    print(f"  |Δ| max   = {np.max(np.abs(diffs_edge)):.5f}")

# Monte Carlo summary
print(f"\n[MONTE CARLO  N={N_MC}]")
print(f"  Successful: {len(ok_recs)}/{N_MC}")
print(f"  |Δ| mean   = {np.mean(np.abs(diff_mc)):.5f}")
print(f"  |Δ| median = {np.median(np.abs(diff_mc)):.5f}")
print(f"  |Δ| max    = {np.max(np.abs(diff_mc)):.5f}")
print(f"  |Δ| < 0.05 : {(np.abs(diff_mc)<0.05).mean()*100:.1f}%")
print(f"  |Δ| < 0.10 : {(np.abs(diff_mc)<0.10).mean()*100:.1f}%")
print(f"  |Δ| < 0.25 : {(np.abs(diff_mc)<0.25).mean()*100:.1f}%")

# Assessment
mean_diff = np.mean(np.abs(diff_mc))
print(f"\n[ASSESSMENT]")
if mean_diff < 0.05:
    verdict = "EXCELLENT — both codes are highly consistent."
elif mean_diff < 0.15:
    verdict = "GOOD — consistent within known approximation differences."
elif mean_diff < 0.4:
    verdict = "MODERATE — notable systematic offset; check angle conventions."
else:
    verdict = "LARGE — significant structural difference; investigate ring-angle mapping."
print(f"  {verdict}")

print("\n[KNOWN SOURCES OF DISCREPANCY]")
print("  1. theta/phir angle mapping: exorings 'theta' is a sky-plane tilt;")
print("     geotrans2 'phir' is a 3-D azimuthal angle projected through iorb.")
print("     For iorb ≈ 90° and e=0, phir ≈ theta is a good approximation.")
print("  2. Density formula: exorings uses Seager (2003) approximation;")
print("     geotrans2.calculate_PR() also uses Seager (2003).")
print("  3. Transit area: exorings uses the analytical approximation;")
print("     geotrans2 uses a full numerical ellipse-intersection engine.")
print("  4. Both codes assume uniform limb-darkening (no LD correction in PR).")
print("=" * 70)


---

## 13 — `geotrans2` PR pipeline with `exorings`-style inputs

Below is a clean, production-ready class that wraps `geotrans2.RingedSystem` and accepts
**exactly the same parameters as `exorings`** (no M★, R★ required).

The stellar mass is fixed at 1 M☉ by default (or passed by the user).
The stellar radius is derived from the mass-radius relation `R★ ∝ M★^0.8`.
All other conversion steps are embedded.


In [ ]:
class ExoringsSystem:
    """
    A geotrans2.RingedSystem wrapper that uses exactly the same
    input parameters as exorings-basic.py.

    Parameters (all keyword, with defaults matching exorings-basic.py)
    ------------------------------------------------------------------
    rhotrue  : float   Stellar density [g/cm^3]             default 1.40598
    P        : float   Orbital period [days]                 default 365.2446
    b        : float   Impact parameter [R*]                 default 0.1875
    p        : float   Planet radius [R*]                    default 0.08
    fi       : float   Ring interior radius [Rp]             default 1.5
    fe       : float   Ring exterior radius [Rp]             default 2.35
    tau      : float   Ring normal opacity                   default 1.0
    theta    : float   Projected tilt [deg]                  default 30.0
    ir       : float   Projected inclination [deg]           default 80.0
    Mstar_solar : float  Stellar mass [Msun]                 default 1.0

    Key methods
    -----------
    .calculate_PR()      → log10(rho_obs / rho_true)
    .system              → the underlying RingedSystem object
    .summary()           → print a compact summary
    """

    def __init__(self,
                 rhotrue=1.40598, P=365.2446, b=0.1875,
                 p=0.08, fi=1.5, fe=2.35,
                 tau=1.0, theta=30.0, ir=80.0,
                 Mstar_solar=1.0):

        self.rhotrue = rhotrue
        self.P       = P
        self.b       = b
        self.p       = p
        self.fi      = fi
        self.fe      = fe
        self.tau     = tau
        self.theta   = theta
        self.ir      = ir

        # ── Stellar properties ────────────────────────────────────────────
        self.Mstar = Mstar_solar * MSUN
        self.Rstar = RSUN * Mstar_solar**0.8
        # Adjust Mstar so density matches rhotrue exactly
        self.Mstar = rhotrue * 1e3 * (4 * np.pi / 3 * self.Rstar**3)

        # ── Derived orbital parameters ────────────────────────────────────
        # Semimajor axis from rhotrue + P (same formula as exorings)
        self.a_over_R = (GCONST * (rhotrue * 1e3) / (3 * np.pi)
                         * (P * DAY)**2)**(1/3)
        self.ap = self.a_over_R * self.Rstar

        # Orbital inclination from impact parameter
        cosiorb = b / self.a_over_R
        if abs(cosiorb) > 1.0:
            raise ValueError(
                f"No transit: impact parameter b={b:.3f} requires "
                f"|cos(iorb)| = {abs(cosiorb):.3f} > 1")
        self.iorb = np.arccos(cosiorb)

        # Planetary radius in metres
        self.Rplanet = p * self.Rstar

        # Ring angles: ir is the ring inclination (same in both codes)
        # phir (geotrans2 azimuthal angle) ≈ theta for iorb ~ 90°
        self.ir_rad   = ir    * DEG
        self.phir_rad = theta * DEG

        # ── Build the RingedSystem ────────────────────────────────────────
        self.system = RingedSystem(dict(
            Mstar   = self.Mstar,
            Rstar   = self.Rstar,
            Mplanet = MSAT,        # mass not used in transit geometry
            Rplanet = self.Rplanet,
            ap      = self.ap,
            ep      = 0.0,
            iorb    = self.iorb,
            wp      = 0.0 * DEG,
            fi      = fi,
            fe      = fe,
            tau     = tau,
            ir      = self.ir_rad,
            phir    = self.phir_rad,
            fp      = 0.0,
        ))

    def calculate_PR(self):
        """Return log10(rho_obs / rho_true)."""
        return self.system.calculate_PR()

    def calculate_rho_obs(self):
        """Return observed stellar density [g/cm^3]."""
        return self.system.calculate_rho_obs()

    def rho_true(self):
        """Return true stellar density [g/cm^3]."""
        return self.system.rho_true

    def summary(self):
        PR = self.calculate_PR()
        print("-" * 55)
        print("  ExoringsSystem — PR Effect Summary")
        print("-" * 55)
        print(f"  rhotrue  = {self.rhotrue:.5f} g/cm³")
        print(f"  P        = {self.P:.4f} days")
        print(f"  b        = {self.b:.4f} R*")
        print(f"  p        = {self.p:.4f} R*")
        print(f"  fi/fe    = {self.fi:.3f} / {self.fe:.3f} Rp")
        print(f"  tau      = {self.tau:.3f}")
        print(f"  theta    = {self.theta:.2f} °")
        print(f"  ir       = {self.ir:.2f} °")
        print(f"  ─────────────────────────────────────")
        print(f"  rho_obs  = {self.system.rho_obs:.5f} g/cm³")
        print(f"  rho_true = {self.system.rho_true:.5f} g/cm³")
        print(f"  log10(PR) = {PR:+.6f}")
        print("-" * 55)
        return PR


# ── Demo ─────────────────────────────────────────────────────────────────────
es = ExoringsSystem()   # all defaults from exorings-basic.py
logPR = es.summary()

# Compare against the exorings analytical result
print(f"\nexorings analytical logPR = {_def['logPR']:+.6f}")
print(f"geotrans2 numerical  logPR = {logPR:+.6f}")
print(f"Difference                  = {abs(logPR - _def['logPR']):.6f}")


### Usage examples for `ExoringsSystem`


In [ ]:
# ── Example 1: Saturn-analogue ────────────────────────────────────────────
print("Saturn-analogue system:")
sat = ExoringsSystem(
    rhotrue=1.40598, P=10760, b=0.05,   # ~29.5-yr orbit
    p=0.0838, fi=1.58, fe=2.35,
    tau=1.0, theta=45.0, ir=75.0
)
sat.summary()

# ── Example 2: hot Jupiter with slight rings ──────────────────────────────
print("\nHot Jupiter with rings:")
hj = ExoringsSystem(
    rhotrue=0.5, P=3.5, b=0.2,
    p=0.12, fi=1.2, fe=2.0,
    tau=0.8, theta=60.0, ir=70.0
)
hj.summary()

# ── Example 3: sweep logPR vs theta for this system ──────────────────────
thetas = np.linspace(1, 89, 30)
logPRs = []
for t in thetas:
    try:
        s = ExoringsSystem(rhotrue=1.40598, P=365.2, b=0.1875,
                           p=0.08, fi=1.5, fe=2.35,
                           tau=1.0, theta=t, ir=80.0)
        logPRs.append(s.calculate_PR())
    except Exception:
        logPRs.append(np.nan)

plt.figure(figsize=(8, 4))
plt.plot(thetas, logPRs, "b-o", ms=4)
plt.xlabel(r"Projected tilt $\theta$ [°]", fontsize=12)
plt.ylabel(r"$\log_{10}(\rho_{\rm obs}/\rho_{\rm true})$", fontsize=12)
plt.title("PR effect vs ring tilt — ExoringsSystem (geotrans2 engine)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
